# Strata Studio on Kaggle

This notebook opens **Strata Studio first**. There is no model picker in this notebook and no `strata launch` step. Studio detects the GPU, installs the matching engine, downloads a model, loads it, and serves it from the browser.

| Cell | What it does |
|---|---|
| **01 - Install WebUI** | Clones Strata and the WebUI under `/kaggle/working`, installs Node 24.12+ if needed, runs `npm ci`, and builds the UI. It downloads no engine and no model. |
| **01b - Prepare Python** | Makes sure `/kaggle/working/Strata/terminal/.venv` has a working Python and pip. If Kaggle's `venv` module is incomplete, it uses a small `virtualenv` bootstrap under `/kaggle/working`; no sudo or apt is needed. |
| **02 - Start Studio** | Starts the real WebUI host on loopback, opens the authenticated Studio gateway, creates a Cloudflare manager tunnel, and shows the Studio key. By default it stays open with periodic health checks; interrupt the cell to return to the notebook without stopping Studio. |
| **03 - Stop** | Stops the manager tunnel, host, engine, and model server. It keeps all files. |

## Storage split

- `/kaggle/working`: source checkouts, WebUI code, Node/npm, Python environment, engine components, run configs, settings, logs, workspace, caches, and cloudflared.
- `/kaggle/tmp`: model weights, prepared model data, and model-download caches only.

Kaggle clears `/kaggle/tmp` when the session ends, so models must be downloaded again in a new session. The WebUI, Python environment, and engine under `/kaggle/working` remain.

## Use it

1. Enable a GPU accelerator if you want GPU inference.
2. Run **01 - Install WebUI** once.
3. Run **01b - Prepare Python** once. If Studio is already open, this can be run on its own before retrying a failed download.
4. Run **02 - Start Studio**. It stays open with best-effort keepalive checks; interrupt the cell to return to the notebook without stopping Studio.
5. Open the private Cloudflare link and paste the **Studio key** shown by the cell.
6. In Studio, follow **Setup** to install the engine, then use **Models** to download, load, tune, unload, delete, or serve a model.

The Studio key controls the manager. It is not the inference API key. Keep the notebook private and clear cell 02's output before saving or sharing it.

In [ ]:
# 01 - Install WebUI
force_node_install = False
source_ref = "main"
webui_ref = "main"
node_version_wanted = "24.12.0"

import hashlib
import html
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import textwrap
import time
import urllib.request
from pathlib import Path
from IPython.display import HTML, display

STRATA_REPO_URL = "https://github.com/Niko1221/Strata.git"
WEBUI_REPO_URL = "https://github.com/GodL-x-SouL/Strata-WebUI.git"
WORKING = Path("/kaggle/working")
MODEL_TMP = Path("/kaggle/tmp")
SUPPORT = WORKING / ".strata-webui"
SOURCE_DIR = WORKING / "Strata"
WEBUI_DIR = WORKING / "Strata-WebUI"
HOME_DIR = SUPPORT / "home"
STATE_DIR = SUPPORT / "state"
COMPONENTS_DIR = SUPPORT / "components"
INSTALL_DIR = SUPPORT / "install"
WORKSPACE_DIR = SUPPORT / "workspace"
XDG_DATA = SUPPORT / "xdg-data"
XDG_STATE = SUPPORT / "xdg-state"
XDG_CONFIG = SUPPORT / "xdg-config"
XDG_CACHE = SUPPORT / "xdg-cache"
TMP_DIR = SUPPORT / "tmp"
TOOLS_DIR = SUPPORT / "tools"
NPM_CACHE = SUPPORT / "npm-cache"
MODEL_DOCUMENTS = MODEL_TMP / "strata-models" / "Documents"
MODELS_DIR = MODEL_DOCUMENTS / "Strata" / "Models"
MODEL_DATA_DIR = MODEL_DOCUMENTS / "Strata"
HF_HOME = MODEL_TMP / "strata-models" / "hf"
TORCH_HOME = MODEL_TMP / "strata-models" / "torch"
RUNTIME_FILE = SUPPORT / "runtime.json"

for directory in (
    SUPPORT, HOME_DIR, STATE_DIR, COMPONENTS_DIR, INSTALL_DIR, WORKSPACE_DIR,
    XDG_DATA, XDG_STATE, XDG_CONFIG, XDG_CACHE, TMP_DIR, TOOLS_DIR, NPM_CACHE,
    MODELS_DIR, MODEL_DATA_DIR, HF_HOME, TORCH_HOME,
):
    directory.mkdir(parents=True, exist_ok=True)
SUPPORT.chmod(0o700)
STATE_DIR.chmod(0o700)

ANSI = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
LOGS = []


def clean(value):
    return ANSI.sub("", str(value)).replace("\r", "").rstrip()


def esc(value):
    return html.escape(str(value) if value is not None else "")


def fmt_bytes(value):
    value = float(value or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if value < 1024 or unit == "TB":
            return "%d %s" % (int(value), unit) if unit == "B" else "%.2f %s" % (value, unit)
        value /= 1024.0


CSS = """
<style>
.st1-wrap{width:100%;box-sizing:border-box;background:#141210;color:#e8dfd0;border:1px solid #3f382c;padding:26px 30px 22px;font-family:Palatino,"Palatino Linotype",Georgia,serif;}
.st1-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}
.st1-kicker,.st1-status,.st1-label{font-family:"Courier New",monospace;font-size:10px;letter-spacing:.22em;color:#b8956c;text-transform:uppercase;}
.st1-title{font-size:27px;color:#f3eadb;margin:7px 0 0;line-height:1.15;}
.st1-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.st1-sub{color:#a89880;font-size:15px;line-height:1.5;}
.st1-WORKING{color:#d2a05a;}.st1-READY{color:#9fb58e;}.st1-ERROR{color:#e18b7d;}.st1-STOPPED{color:#a89880;}
.st1-row{display:flex;justify-content:space-between;gap:18px;padding:5px 0;font-size:15px;line-height:1.4;}
.st1-key{color:#8a8070;min-width:150px;}.st1-value{color:#e8dfd0;text-align:right;word-break:break-word;}
.st1-bar{height:6px;background:#2a261f;border:1px solid #3f382c;margin:14px 0 8px;overflow:hidden;}
.st1-fill{height:100%;background:linear-gradient(90deg,#8a6a3c,#d2a05a);width:0%;transition:width .2s;}
.st1-fill.indeterminate{width:40%!important;animation:st1slide 1.1s ease-in-out infinite alternate;}
@keyframes st1slide{from{margin-left:0}to{margin-left:60%}}
.st1-stat{font-family:"Courier New",monospace;font-size:11px;color:#a89880;}
.st1-console{margin-top:16px;border:1px solid #3f382c;background:#0e0c0a;}
.st1-console pre{margin:0;padding:12px 14px;font-family:"Courier New",monospace;font-size:11px;line-height:1.45;color:#bdb3a0;white-space:pre-wrap;word-break:break-word;max-height:280px;overflow:auto;}
.st1-foot{margin-top:15px;font-family:"Courier New",monospace;font-size:11px;color:#8a8070;line-height:1.5;}
</style>
"""


class Card:
    def __init__(self):
        self.status = "WORKING"
        self.sub = "Preparing the WebUI checkout."
        self.rows = []
        self.foot = "No engine or model is downloaded by this cell."
        self.download_name = ""
        self.done = 0
        self.total = 0
        self.handle = None
        self.last_paint = 0.0
        self.paint()

    def paint(self):
        progress = ""
        if self.download_name:
            percent = 0 if not self.total else min(100.0, self.done * 100.0 / self.total)
            width = "%.2f%%" % percent if self.total else "40%"
            klass = "" if self.total else " indeterminate"
            progress = (
                '<div class="st1-label">%s</div><div class="st1-bar"><div class="st1-fill%s" style="width:%s"></div></div>'
                '<div class="st1-stat">%s / %s · %.1f%%</div>'
                % (esc(self.download_name), klass, width, fmt_bytes(self.done),
                   fmt_bytes(self.total) if self.total else "-", percent)
            )
        rows = "".join(
            '<div class="st1-row"><span class="st1-key">%s</span><span class="st1-value">%s</span></div>'
            % (esc(key), esc(value)) for key, value in self.rows[-10:]
        )
        console = "".join(esc(line) + "\n" for line in LOGS[-14:])
        output = HTML(CSS + """
        <div class="st1-wrap">
          <div class="st1-top"><div class="st1-kicker">01 / Install WebUI</div><div class="st1-status st1-%s">%s</div></div>
          <div class="st1-title">Strata Studio on Kaggle</div><hr class="st1-rule">
          <div class="st1-sub">%s</div>%s
          <div style="margin-top:14px">%s</div>
          <div class="st1-console"><pre>%s</pre></div>
          <div class="st1-foot">%s</div>
        </div>""" % (esc(self.status), esc(self.status), esc(self.sub), progress, rows, console, esc(self.foot)))
        if self.handle is None:
            self.handle = display(output, display_id="st1-install")
        else:
            self.handle.update(output)
        self.last_paint = time.monotonic()

    def set(self, status=None, sub=None, rows=None, foot=None):
        if status is not None:
            self.status = status
        if sub is not None:
            self.sub = sub
        if rows is not None:
            self.rows = rows
        if foot is not None:
            self.foot = foot
        self.paint()

    def log(self, tag, message):
        line = clean(message)
        if not line:
            return
        line = textwrap.shorten("%s [%s] %s" % (time.strftime("%H:%M:%S"), tag, line), width=1200)
        LOGS.append(line)
        del LOGS[:-300]
        if time.monotonic() - self.last_paint >= 0.15:
            self.paint()

    def progress(self, name, done, total):
        self.download_name = name
        self.done = done
        self.total = total
        if time.monotonic() - self.last_paint >= 0.12:
            self.paint()


ui = Card()


def run_logged(command, cwd=None, env=None, tag="RUN"):
    ui.log(tag, " ".join(str(part) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command], cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding="utf-8", errors="replace", bufsize=1,
    )
    try:
        for line in iter(process.stdout.readline, ""):
            ui.log(tag, line)
        code = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
        raise
    if code != 0:
        raise RuntimeError("%s exited with code %s. See the console." % (command[0], code))
    ui.paint()


def download(url, target, timeout=180, attempts=3):
    target = Path(target)
    temporary = target.with_name(target.name + ".part")
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "strata-kaggle-webui/1.0"})
            with urllib.request.urlopen(request, timeout=timeout) as response, temporary.open("wb") as output:
                total = int(response.headers.get("Content-Length") or 0)
                done = 0
                while True:
                    chunk = response.read(1024 * 256)
                    if not chunk:
                        break
                    output.write(chunk)
                    done += len(chunk)
                    ui.progress(target.name, done, total)
            os.replace(temporary, target)
            ui.download_name = ""
            ui.paint()
            return target
        except Exception as error:
            last_error = error
            temporary.unlink(missing_ok=True)
            ui.log("DOWNLOAD", "attempt %d/%d failed: %s" % (attempt, attempts, clean(error)))
            if attempt < attempts:
                time.sleep(2 * attempt)
    raise RuntimeError("Could not download %s: %s" % (url, clean(last_error)))


def version_tuple(text):
    match = re.search(r"v?(\d+)\.(\d+)\.(\d+)", str(text))
    return tuple(int(part) for part in match.groups()) if match else ()


def executable_version(path):
    try:
        result = subprocess.run([str(path), "--version"], capture_output=True, text=True, timeout=20)
        return version_tuple((result.stdout or result.stderr).strip())
    except Exception:
        return ()


def ensure_repo(url, directory, ref):
    if directory.exists() and not (directory / ".git").is_dir():
        raise RuntimeError("%s exists but is not a Git checkout. Move it aside; this notebook will not delete it." % directory)
    if not (directory / ".git").is_dir():
        run_logged(["git", "clone", "--depth", "1", "--branch", ref, url, directory], tag="GIT")
        return
    dirty = subprocess.run(
        ["git", "-C", str(directory), "status", "--porcelain", "--untracked-files=no"],
        capture_output=True, text=True, check=False,
    )
    if dirty.stdout.strip():
        raise RuntimeError("%s has local tracked changes. Move or commit them before updating." % directory)
    run_logged(["git", "-C", directory, "fetch", "--depth", "1", "origin", ref], tag="GIT")
    run_logged(["git", "-C", directory, "checkout", ref], tag="GIT")
    run_logged(["git", "-C", directory, "merge", "--ff-only", "FETCH_HEAD"], tag="GIT")


def node_install():
    machine = platform_machine = os.uname().machine.lower()
    arch = {"x86_64": "x64", "amd64": "x64", "aarch64": "arm64", "arm64": "arm64"}.get(machine)
    if not arch:
        raise RuntimeError("Unsupported Linux architecture for the Node download: %s" % machine)
    folder = TOOLS_DIR / ("node-v%s-linux-%s" % (node_version_wanted, arch))
    candidate = folder / "bin" / "node"
    if candidate.is_file() and executable_version(candidate) >= version_tuple(node_version_wanted):
        return candidate
    filename = "node-v%s-linux-%s.tar.xz" % (node_version_wanted, arch)
    base_url = "https://nodejs.org/dist/v%s/" % node_version_wanted
    archive = download(base_url + filename, TOOLS_DIR / filename, timeout=600)
    checksums = download(base_url + "SHASUMS256.txt", TOOLS_DIR / "SHASUMS256.txt", timeout=90)
    wanted = None
    for line in checksums.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) >= 2 and parts[1].lstrip("*") == filename:
            wanted = parts[0]
            break
    if not wanted:
        raise RuntimeError("The Node checksum file does not list %s." % filename)
    digest = hashlib.sha256()
    with archive.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != wanted:
        raise RuntimeError("The Node archive checksum does not match SHASUMS256.txt.")
    ui.set(sub="Verifying and unpacking Node %s." % node_version_wanted)
    with tarfile.open(archive, "r:xz") as bundle:
        try:
            bundle.extractall(TOOLS_DIR, filter="data")
        except TypeError:
            bundle.extractall(TOOLS_DIR)
    if not candidate.is_file():
        raise RuntimeError("The Node archive did not contain %s." % candidate)
    return candidate


def gpu_rows():
    nvidia_smi = shutil.which("nvidia-smi")
    if not nvidia_smi:
        return ["No NVIDIA GPU was detected. Studio will still open and report what this machine supports."]
    command = [
        nvidia_smi, "--query-gpu=index,name,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ]
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=30)
        if result.returncode != 0:
            return ["nvidia-smi could not read the GPU: " + clean(result.stderr)[:300]]
        return [line.strip() for line in result.stdout.splitlines() if line.strip()]
    except Exception as error:
        return ["GPU probe failed: " + clean(error)]


try:
    if sys.version_info < (3, 10):
        raise RuntimeError("Python 3.10 or newer is required. This kernel is %s." % sys.version.split()[0])
    if not shutil.which("git"):
        raise RuntimeError("Git is missing from this Kaggle image.")

    ui.set(sub="Checking Node and the two source checkouts.", rows=[("support", str(SUPPORT)), ("models", str(MODELS_DIR))])
    current_node = shutil.which("node")
    required_node = version_tuple(node_version_wanted)
    if force_node_install or not current_node or executable_version(current_node) < required_node:
        ui.set(sub="Kaggle's Node is missing or too old. Installing Node %s under /kaggle/working." % node_version_wanted)
        node_bin = node_install()
    else:
        node_bin = Path(current_node)
    node_version = subprocess.run([str(node_bin), "--version"], capture_output=True, text=True, check=True).stdout.strip()
    npm_bin = node_bin.parent / "npm"
    if not npm_bin.is_file():
        npm_bin = Path(shutil.which("npm") or "")
    if not npm_bin.is_file():
        raise RuntimeError("npm was not found beside Node.")

    ui.set(sub="Updating Strata and the WebUI under /kaggle/working.")
    ensure_repo(STRATA_REPO_URL, SOURCE_DIR, source_ref)
    ensure_repo(WEBUI_REPO_URL, WEBUI_DIR, webui_ref)
    for required in (
        SOURCE_DIR / "terminal" / "setup.py",
        SOURCE_DIR / "terminal" / "tools" / "strata_mcp.py",
        SOURCE_DIR / "terminal" / "serve" / "server.py",
        WEBUI_DIR / "server" / "index.mjs",
        WEBUI_DIR / "package.json",
        WEBUI_DIR / "package-lock.json",
    ):
        if not required.is_file():
            raise RuntimeError("The checkout is incomplete: %s is missing." % required)

    build_env = os.environ.copy()
    for key in ("PYTHONHOME", "PYTHONPATH", "VIRTUAL_ENV", "STRATA_MODELS_DIR", "STRATA_DATA_DIR", "STRATA_CONTROL_DIR", "STRATA_PACKAGED", "STRATA_NO_ACQUIRE", "STRATA_TEST_HARDWARE"):
        build_env.pop(key, None)
    build_env.update({
        "PATH": str(node_bin.parent) + os.pathsep + build_env.get("PATH", ""),
        "HOME": str(HOME_DIR),
        "TMPDIR": str(TMP_DIR),
        "XDG_DATA_HOME": str(XDG_DATA),
        "XDG_STATE_HOME": str(XDG_STATE),
        "XDG_CONFIG_HOME": str(XDG_CONFIG),
        "XDG_CACHE_HOME": str(XDG_CACHE),
        "NPM_CONFIG_CACHE": str(NPM_CACHE),
        "PIP_CACHE_DIR": str(XDG_CACHE / "pip"),
        "UV_CACHE_DIR": str(XDG_CACHE / "uv"),
        "NO_COLOR": "1",
        "PYTHONUNBUFFERED": "1",
        "PYTHONUTF8": "1",
        "PYTHONIOENCODING": "utf-8",
    })
    ui.set(sub="Installing the WebUI's locked Node dependencies.")
    run_logged([npm_bin, "ci", "--no-audit", "--no-fund"], cwd=WEBUI_DIR, env=build_env, tag="NPM")
    ui.set(sub="Building the WebUI.")
    run_logged([npm_bin, "run", "build"], cwd=WEBUI_DIR, env=build_env, tag="BUILD")
    if not (WEBUI_DIR / "dist" / "index.html").is_file():
        raise RuntimeError("The build finished without dist/index.html.")

    runtime_env = {
        key: build_env[key]
        for key in (
            "PATH", "HOME", "TMPDIR",
            "XDG_DATA_HOME", "XDG_STATE_HOME", "XDG_CONFIG_HOME", "XDG_CACHE_HOME",
            "NPM_CONFIG_CACHE", "PIP_CACHE_DIR", "UV_CACHE_DIR",
            "LANG", "LANGUAGE", "LC_ALL", "LC_CTYPE", "TZ", "NO_COLOR",
            "PYTHONUNBUFFERED", "PYTHONUTF8", "PYTHONIOENCODING",
            "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS",
            "LD_LIBRARY_PATH", "CUDA_HOME", "CUDA_PATH", "ROCM_HOME", "HIP_PATH",
            "CUDA_VISIBLE_DEVICES", "NVIDIA_VISIBLE_DEVICES", "HIP_VISIBLE_DEVICES", "ROCR_VISIBLE_DEVICES",
            "HF_HUB_DISABLE_TELEMETRY", "DO_NOT_TRACK",
        )
        if key in build_env
    }
    runtime_env.update({
        "STRATA_HOME": str(STATE_DIR),
        "STRATA_NATIVE_REPO": str(SOURCE_DIR),
        "STRATA_PYTHON": sys.executable,
        "STRATA_DOCUMENTS_DIR": str(MODEL_DOCUMENTS),
        "STRATA_COMPONENTS_DIR": str(COMPONENTS_DIR),
        "STRATA_INSTALL_DIR": str(INSTALL_DIR),
        "STRATA_WORKSPACE": str(WORKSPACE_DIR),
        "STRATA_SETTINGS_FILE": str(SUPPORT / "native-settings.json"),
        "STRATA_DIST": str(WEBUI_DIR / "dist"),
        "STRATA_BIND": "127.0.0.1",
        "STRATA_STUDIO": "1",
        "STRATA_REAL_ENGINE": "1",
        "STRATA_EXIT_WITH_PARENT": "1",
        "HF_HOME": str(HF_HOME),
        "HF_HUB_CACHE": str(HF_HOME / "hub"),
        "TORCH_HOME": str(TORCH_HOME),
        "TRITON_CACHE_DIR": str(XDG_CACHE / "triton"),
    })
    runtime_env.pop("STRATA_MODELS_DIR", None)
    runtime_env.pop("STRATA_DATA_DIR", None)

    runtime = {
        "version": 1,
        "working": str(WORKING),
        "support": str(SUPPORT),
        "source": str(SOURCE_DIR),
        "webui": str(WEBUI_DIR),
        "dist": str(WEBUI_DIR / "dist"),
        "node": str(node_bin),
        "npm": str(npm_bin),
        "python": sys.executable,
        "models": str(MODELS_DIR),
        "model_data": str(MODEL_DATA_DIR),
        "hf_home": str(HF_HOME),
        "env": runtime_env,
        "preferred_ports": {"host": 8791, "studio": 8794, "share": 8795, "tunnel": 8796},
        "installed_at": int(time.time()),
    }
    temporary_runtime = RUNTIME_FILE.with_suffix(".json.tmp")
    temporary_runtime.write_text(json.dumps(runtime, indent=2), encoding="utf-8")
    temporary_runtime.chmod(0o600)
    os.replace(temporary_runtime, RUNTIME_FILE)

    working_free = shutil.disk_usage(WORKING).free / 1024 ** 3
    model_free = shutil.disk_usage(MODEL_TMP).free / 1024 ** 3
    gpu = gpu_rows()
    rows = [
        ("Node", node_version),
        ("Python", sys.version.split()[0]),
        ("Strata source", str(SOURCE_DIR)),
        ("WebUI", str(WEBUI_DIR)),
        ("engine + settings", str(SUPPORT)),
        ("models", str(MODELS_DIR)),
        ("working free", "%.1f GB" % working_free),
        ("model disk free", "%.1f GB" % model_free),
    ]
    rows.extend(("GPU %d" % (index + 1), value) for index, value in enumerate(gpu))
    warnings = []
    if working_free < 3:
        warnings.append("Less than 3 GB remains in /kaggle/working for the engine and build files.")
    if model_free < 8:
        warnings.append("Less than 8 GB remains in /kaggle/tmp for models.")
    ui.set(
        status="READY",
        sub="The WebUI is built. No engine or model was downloaded. Run 01b - Prepare Python, then 02 - Start Studio.",
        rows=rows,
        foot=" ".join(warnings) if warnings else "Studio will install the engine and models only after you choose them in the browser.",
    )
    ui.log("READY", str(RUNTIME_FILE))
except KeyboardInterrupt:
    ui.set(status="STOPPED", sub="Installation interrupted. Run this cell again to continue.", foot="Completed downloads and checkouts were kept.")
except Exception as error:
    ui.set(status="ERROR", sub=clean(error) or error.__class__.__name__, foot="Fix the issue above, then run this cell again. No model was downloaded.")


In [ ]:
# 01b - Prepare Python
import html
import os
import re
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path
from IPython.display import HTML, display

WORKING = Path("/kaggle/working")
SUPPORT = WORKING / ".strata-webui"
SOURCE_DIR = WORKING / "Strata"
TERMINAL_DIR = SOURCE_DIR / "terminal"
VENV_DIR = TERMINAL_DIR / ".venv"
VENV_PYTHON = VENV_DIR / "bin" / "python"
BOOTSTRAP_DIR = SUPPORT / "tools" / "virtualenv"
HOME_DIR = SUPPORT / "home"
TMP_DIR = SUPPORT / "tmp"
CACHE_DIR = SUPPORT / "xdg-cache"
ANSI = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
LOGS = []


def clean(value):
    return ANSI.sub("", str(value)).replace("\r", "").rstrip()


def esc(value):
    return html.escape(str(value) if value is not None else "")


CSS = """
<style>
.st1b-wrap{width:100%;box-sizing:border-box;background:#141210;color:#e8dfd0;border:1px solid #3f382c;padding:26px 30px 22px;font-family:Palatino,"Palatino Linotype",Georgia,serif;}
.st1b-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}
.st1b-kicker,.st1b-status,.st1b-label{font-family:"Courier New",monospace;font-size:10px;letter-spacing:.22em;color:#b8956c;text-transform:uppercase;}
.st1b-title{font-size:27px;color:#f3eadb;margin:7px 0 0;line-height:1.15;}
.st1b-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.st1b-sub{color:#a89880;font-size:15px;line-height:1.5;}
.st1b-WORKING{color:#d2a05a;}.st1b-READY{color:#9fb58e;}.st1b-ERROR{color:#e18b7d;}.st1b-STOPPED{color:#a89880;}
.st1b-row{display:flex;justify-content:space-between;gap:18px;padding:5px 0;font-size:15px;line-height:1.4;}
.st1b-key{color:#8a8070;min-width:150px;}.st1b-value{color:#e8dfd0;text-align:right;word-break:break-word;}
.st1b-console{margin-top:16px;border:1px solid #3f382c;background:#0e0c0a;}
.st1b-console pre{margin:0;padding:12px 14px;font-family:"Courier New",monospace;font-size:11px;line-height:1.45;color:#bdb3a0;white-space:pre-wrap;word-break:break-word;max-height:260px;overflow:auto;}
.st1b-foot{margin-top:15px;font-family:"Courier New",monospace;font-size:11px;color:#8a8070;line-height:1.5;}
</style>
"""


class Card:
    def __init__(self):
        self.status = "WORKING"
        self.sub = "Checking Strata's Python environment."
        self.rows = []
        self.foot = "No engine or model is downloaded by this cell."
        self.handle = None
        self.paint()

    def paint(self):
        rows = "".join(
            '<div class="st1b-row"><span class="st1b-key">%s</span><span class="st1b-value">%s</span></div>'
            % (esc(name), esc(value)) for name, value in self.rows
        )
        console = "".join(esc(line) + "\n" for line in LOGS[-18:])
        output = HTML(CSS + """
        <div class="st1b-wrap">
          <div class="st1b-top"><div class="st1b-kicker">01b / Prepare Python</div><div class="st1b-status st1b-%s">%s</div></div>
          <div class="st1b-title">Python environment</div><hr class="st1b-rule">
          <div class="st1b-sub">%s</div>
          <div style="margin-top:14px">%s</div>
          <div class="st1b-console"><pre>%s</pre></div>
          <div class="st1b-foot">%s</div>
        </div>""" % (esc(self.status), esc(self.status), esc(self.sub), rows, console, esc(self.foot)))
        if self.handle is None:
            self.handle = display(output, display_id="st1b-python")
        else:
            self.handle.update(output)

    def set(self, status=None, sub=None, rows=None, foot=None):
        if status is not None:
            self.status = status
        if sub is not None:
            self.sub = sub
        if rows is not None:
            self.rows = rows
        if foot is not None:
            self.foot = foot
        self.paint()

    def log(self, message):
        line = clean(message)
        if line:
            LOGS.append(textwrap.shorten("%s [PYTHON] %s" % (time.strftime("%H:%M:%S"), line), width=1200))
            del LOGS[:-200]
            self.paint()


ui = Card()


def run_logged(command, env, tag):
    ui.log("%s: %s" % (tag, " ".join(str(part) for part in command)))
    process = subprocess.Popen(
        [str(part) for part in command], env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding="utf-8", errors="replace", bufsize=1,
    )
    try:
        for line in iter(process.stdout.readline, ""):
            ui.log(line)
        return process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
        raise


def command_works(command, env=None):
    try:
        result = subprocess.run(
            [str(part) for part in command], env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", timeout=30,
        )
        return result.returncode == 0, clean(result.stdout)
    except Exception as error:
        return False, clean(error)


def environment_ready():
    ok, detail = command_works([
        VENV_PYTHON, "-c",
        "import sys; assert sys.version_info >= (3, 10); import pip",
    ])
    if not ok:
        return False
    ok, _ = command_works([VENV_PYTHON, "-m", "pip", "--version"])
    return ok


def support_environment():
    for directory in (SUPPORT, HOME_DIR, TMP_DIR, CACHE_DIR, BOOTSTRAP_DIR):
        directory.mkdir(parents=True, exist_ok=True)
    SUPPORT.chmod(0o700)
    env = os.environ.copy()
    for key in ("PYTHONHOME", "PYTHONPATH", "VIRTUAL_ENV", "PIP_REQUIRE_VIRTUALENV"):
        env.pop(key, None)
    env.update({
        "HOME": str(HOME_DIR),
        "TMPDIR": str(TMP_DIR),
        "XDG_CACHE_HOME": str(CACHE_DIR),
        "PIP_CACHE_DIR": str(CACHE_DIR / "pip"),
        "PYTHONUNBUFFERED": "1",
        "PYTHONUTF8": "1",
        "PYTHONIOENCODING": "utf-8",
        "PYTHONNOUSERSITE": "1",
    })
    return env


try:
    if not (TERMINAL_DIR / "setup.py").is_file():
        raise RuntimeError("Run 01 - Install WebUI first. %s is missing." % (TERMINAL_DIR / "setup.py"))
    ui.set(rows=[("environment", str(VENV_DIR)), ("Python", sys.version.split()[0])])
    if environment_ready():
        version = subprocess.run(
            [str(VENV_PYTHON), "--version"], capture_output=True, text=True, check=True,
        ).stdout.strip()
        ui.set(status="READY", sub="Strata's Python environment is ready.", rows=[
            ("environment", str(VENV_DIR)), ("Python", version), ("pip", "available"),
        ], foot="Run 02 - Start Studio, or return to the browser and retry the model download.")
    else:
        if VENV_DIR.exists():
            backup = VENV_DIR.with_name(".venv.broken-%d" % time.time_ns())
            VENV_DIR.rename(backup)
            ui.log("Preserved the incomplete environment at %s" % backup)
        env = support_environment()
        ui.set(sub="Trying Python's standard venv module.")
        code = run_logged([sys.executable, "-m", "venv", str(VENV_DIR)], env, "VENV")
        if code == 0 and environment_ready():
            ui.log("The standard venv module created a working environment.")
        else:
            ui.log("The standard venv module is incomplete; using virtualenv instead.")
            if VENV_DIR.exists():
                shutil.rmtree(VENV_DIR)
            pip_ok, pip_detail = command_works([sys.executable, "-m", "pip", "--version"], env)
            if not pip_ok:
                raise RuntimeError("Kaggle's Python has no working pip, so the environment cannot be prepared: %s" % pip_detail)
            env["PYTHONPATH"] = str(BOOTSTRAP_DIR)
            virtualenv_ok, _ = command_works([sys.executable, "-m", "virtualenv", "--version"], env)
            if not virtualenv_ok:
                ui.set(sub="Installing the small virtualenv bootstrap under /kaggle/working.")
                code = run_logged([
                    sys.executable, "-m", "pip", "install",
                    "--disable-pip-version-check", "--no-input", "--upgrade",
                    "--target", str(BOOTSTRAP_DIR), "virtualenv",
                ], env, "PIP")
                if code != 0:
                    raise RuntimeError("Could not install virtualenv. Check the network and retry.")
            ui.set(sub="Creating Strata's pip-enabled Python environment without sudo.")
            code = run_logged([
                sys.executable, "-m", "virtualenv",
                "--no-periodic-update", "--no-download", str(VENV_DIR),
            ], env, "VIRTUALENV")
            if code != 0 or not environment_ready():
                raise RuntimeError("virtualenv did not create a working pip environment. See the console.")
        version = subprocess.run(
            [str(VENV_PYTHON), "--version"], capture_output=True, text=True, check=True,
        ).stdout.strip()
        ui.set(status="READY", sub="Strata's Python environment is ready.", rows=[
            ("environment", str(VENV_DIR)), ("Python", version), ("pip", "available"),
            ("bootstrap", str(BOOTSTRAP_DIR)),
        ], foot="Return to Studio and retry the download. If Studio is already open, it will use this environment on the next install attempt.")
except KeyboardInterrupt:
    ui.set(status="STOPPED", sub="Python preparation was interrupted.", foot="Completed files were kept. Run this cell again to continue.")
except Exception as error:
    ui.set(status="ERROR", sub=clean(error) or error.__class__.__name__, rows=[
        ("environment", str(VENV_DIR)), ("source", str(TERMINAL_DIR)),
    ], foot="No engine or model was downloaded. Fix the error above, then run this cell again.")


In [ ]:
# 02 - Start Studio
open_tunnel = True
tunnel_wait_seconds = 150
keep_alive = True
keep_alive_interval_seconds = 300

import html
import http.cookiejar
import json
import os
import re
import shutil
import signal
import socket
import subprocess
import textwrap
import time
import urllib.error
import urllib.request
import uuid
from pathlib import Path
from IPython.display import HTML, display

RUNTIME_FILE = Path("/kaggle/working/.strata-webui/runtime.json")
ANSI = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
LOGS = []


def clean(value):
    return ANSI.sub("", str(value)).replace("\r", "").rstrip()


def esc(value):
    return html.escape(str(value) if value is not None else "")


CSS = """
<style>
.st2-wrap{width:100%;box-sizing:border-box;background:#141210;color:#e8dfd0;border:1px solid #3f382c;padding:26px 30px 22px;font-family:Palatino,"Palatino Linotype",Georgia,serif;}
.st2-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}
.st2-kicker,.st2-status,.st2-label{font-family:"Courier New",monospace;font-size:10px;letter-spacing:.22em;color:#b8956c;text-transform:uppercase;}
.st2-title{font-size:27px;color:#f3eadb;margin:7px 0 0;line-height:1.15;}
.st2-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.st2-sub{color:#a89880;font-size:15px;line-height:1.5;}
.st2-WORKING{color:#d2a05a;}.st2-LIVE{color:#9fb58e;}.st2-ERROR{color:#e18b7d;}.st2-STOPPED{color:#a89880;}
.st2-url{margin:18px 0;padding:17px 18px;background:#0e0c0a;border:1px solid #5a4e3c;text-align:center;}
.st2-url a{font-family:"Courier New",monospace;font-size:17px;color:#f3eadb;text-decoration:none;border-bottom:1px solid #5a4e3c;word-break:break-all;}
.st2-empty{margin:18px 0;padding:15px;border:1px dashed #5a4e3c;color:#a89880;text-align:center;}
.st2-key{margin:16px 0;padding:14px 16px;background:#0e0c0a;border:1px solid #5a4e3c;}
.st2-key code{display:block;margin-top:7px;font-family:"Courier New",monospace;font-size:14px;color:#f3eadb;word-break:break-all;user-select:text;}
.st2-row{display:flex;justify-content:space-between;gap:18px;padding:5px 0;font-size:15px;line-height:1.4;}
.st2-key-label{color:#8a8070;min-width:150px;}.st2-value{color:#e8dfd0;text-align:right;word-break:break-word;}
.st2-console{margin-top:16px;border:1px solid #3f382c;background:#0e0c0a;}
.st2-console pre{margin:0;padding:12px 14px;font-family:"Courier New",monospace;font-size:11px;line-height:1.45;color:#bdb3a0;white-space:pre-wrap;word-break:break-word;max-height:260px;overflow:auto;}
.st2-foot{margin-top:15px;font-family:"Courier New",monospace;font-size:11px;color:#8a8070;line-height:1.5;}
</style>
"""


class Card:
    def __init__(self):
        self.status = "WORKING"
        self.sub = "Reading the installed WebUI runtime."
        self.rows = []
        self.foot = "The host listens on loopback. Only the authenticated Studio gateway is tunneled."
        self.url = ""
        self.key = ""
        self.handle = None
        self.paint()

    def paint(self):
        if self.url:
            address = '<div class="st2-url"><div class="st2-label">Private Studio link</div><a href="%s" target="_blank" rel="noopener noreferrer">%s</a></div>' % (esc(self.url), esc(self.url))
        else:
            address = '<div class="st2-empty">The private Cloudflare link is not connected yet.</div>'
        key = ""
        if self.key:
            key = '<div class="st2-key"><div class="st2-label">Studio key · manager access only</div><code>%s</code></div>' % esc(self.key)
        rows = "".join(
            '<div class="st2-row"><span class="st2-key-label">%s</span><span class="st2-value">%s</span></div>'
            % (esc(name), esc(value)) for name, value in self.rows
        )
        console_lines = list(LOGS[-8:])
        log_path = globals().get("HOST_LOG")
        if log_path and Path(log_path).is_file():
            try:
                console_lines.extend(clean(line) for line in Path(log_path).read_text(encoding="utf-8", errors="replace").splitlines()[-12:] if clean(line))
            except OSError:
                pass
        console = "".join(esc(line) + "\n" for line in console_lines[-18:])
        output = HTML(CSS + """
        <div class="st2-wrap">
          <div class="st2-top"><div class="st2-kicker">02 / Start Studio</div><div class="st2-status st2-%s">%s</div></div>
          <div class="st2-title">Strata Studio</div><hr class="st2-rule">
          <div class="st2-sub">%s</div>%s%s
          <div style="margin-top:14px">%s</div>
          <div class="st2-console"><pre>%s</pre></div>
          <div class="st2-foot">%s</div>
        </div>""" % (esc(self.status), esc(self.status), esc(self.sub), address, key, rows, console, esc(self.foot)))
        if self.handle is None:
            self.handle = display(output, display_id="st2-studio")
        else:
            self.handle.update(output)

    def set(self, status=None, sub=None, rows=None, foot=None, url=None, key=None):
        if status is not None:
            self.status = status
        if sub is not None:
            self.sub = sub
        if rows is not None:
            self.rows = rows
        if foot is not None:
            self.foot = foot
        if url is not None:
            self.url = url
        if key is not None:
            self.key = key
        self.paint()

    def log(self, tag, message):
        line = clean(message)
        if line:
            LOGS.append(textwrap.shorten("%s [%s] %s" % (time.strftime("%H:%M:%S"), tag, line), width=1200))
            del LOGS[:-200]
            self.paint()


ui = Card()
LOCAL_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


def request_json(url, body=None, headers=None, opener=None, timeout=20):
    request_headers = {"User-Agent": "strata-kaggle-webui/1.0"}
    data = None
    if body is not None:
        data = json.dumps(body).encode("utf-8")
        request_headers["Content-Type"] = "application/json"
    request_headers.update(headers or {})
    if request_headers.get("Origin"):
        request_headers["Origin"] = request_headers["Origin"].rstrip("/")
    request = urllib.request.Request(url, data=data, headers=request_headers, method="POST" if body is not None else "GET")
    try:
        with (opener or LOCAL_OPENER).open(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8", "replace"))
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8", "replace")
        try:
            detail = json.loads(detail).get("error", detail)
        except Exception:
            pass
        raise RuntimeError("Studio returned HTTP %s: %s" % (error.code, clean(detail))) from error


def pick_port(preferred, used):
    for port in range(preferred, min(preferred + 200, 65536)):
        if port in used:
            continue
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
            try:
                probe.bind(("127.0.0.1", port))
                used.add(port)
                return port
            except OSError:
                continue
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        probe.bind(("127.0.0.1", 0))
        port = probe.getsockname()[1]
    used.add(port)
    return port


def read_host_record():
    try:
        return json.loads(HOST_FILE.read_text(encoding="utf-8"))
    except Exception:
        return None


def write_host_record(record):
    temporary = HOST_FILE.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(record, indent=2), encoding="utf-8")
    temporary.chmod(0o600)
    os.replace(temporary, HOST_FILE)


def process_alive(pid):
    try:
        stat = Path("/proc/%d/stat" % int(pid)).read_text(encoding="utf-8")
        if stat.rsplit(")", 1)[1].split()[0] == "Z":
            return False
        os.kill(int(pid), 0)
        return True
    except (OSError, ValueError, TypeError, IndexError):
        return False


def owned_process(record):
    if not record or not record.get("pid") or not record.get("run_id"):
        return False
    pid = int(record["pid"])
    if not process_alive(pid):
        return False
    try:
        command = Path("/proc/%d/cmdline" % pid).read_bytes().replace(b"\0", b" ").decode("utf-8", "replace")
        environment = Path("/proc/%d/environ" % pid).read_bytes().split(b"\0")
        cwd = Path("/proc/%d/cwd" % pid).resolve()
        return (
            "server/index.mjs" in command
            and cwd == Path(record["webui"]).resolve()
            and ("STRATA_KAGGLE_RUN_ID=" + record["run_id"]).encode() in environment
        )
    except Exception:
        return False


def wait_for_studio(studio_url, process=None, timeout=75):
    studio_url = studio_url.rstrip("/")
    deadline = time.monotonic() + timeout
    last_error = ""
    while time.monotonic() < deadline:
        if process is not None and process.poll() is not None:
            raise RuntimeError("The WebUI host exited with code %s. See the console." % process.returncode)
        try:
            return request_json(studio_url + "/studio/local-key", timeout=3)["key"]
        except Exception as error:
            last_error = clean(error)
            time.sleep(1)
    raise RuntimeError("Studio did not open its local gateway: %s" % (last_error or "timed out"))


def login(studio_url, key):
    studio_url = studio_url.rstrip("/")
    jar = http.cookiejar.CookieJar()
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}), urllib.request.HTTPCookieProcessor(jar))
    request_json(studio_url + "/studio/login", {"key": key}, {"Origin": studio_url}, opener=opener, timeout=20)
    return opener


def connect_manager_tunnel(studio_url, key):
    studio_url = studio_url.rstrip("/")
    opener = login(studio_url, key)
    state = request_json(studio_url + "/api/studio", opener=opener, timeout=10)
    tunnel = state.get("tunnel") or {}
    if tunnel.get("status") == "connected" and tunnel.get("url"):
        return tunnel["url"]
    stop_deadline = time.monotonic() + 30
    while tunnel.get("status") == "stopping":
        if time.monotonic() >= stop_deadline:
            raise RuntimeError("The previous Studio tunnel is still stopping. Retry in a moment.")
        ui.set(sub="Waiting for the previous Studio tunnel to stop.")
        time.sleep(2)
        state = request_json(studio_url + "/api/studio", opener=opener, timeout=10)
        tunnel = state.get("tunnel") or {}
    if tunnel.get("status") == "connected" and tunnel.get("url"):
        return tunnel["url"]
    if tunnel.get("status") != "starting":
        ui.log("TUNNEL", "Starting the authenticated Studio Quick Tunnel.")
        state = request_json(
            studio_url + "/api/studio/tunnel",
            {"action": "start", "mode": "quick"},
            {"Origin": studio_url}, opener=opener, timeout=600,
        )
        tunnel = state.get("tunnel") or {}
    deadline = time.monotonic() + tunnel_wait_seconds
    while time.monotonic() < deadline:
        status = tunnel.get("status")
        if status == "connected" and tunnel.get("url"):
            return tunnel["url"]
        if status == "error":
            raise RuntimeError(tunnel.get("error") or "Cloudflare could not connect the Studio tunnel.")
        ui.set(sub="Cloudflare is connecting the private Studio link. This can take up to a minute.")
        time.sleep(2)
        state = request_json(studio_url + "/api/studio", opener=opener, timeout=10)
        tunnel = state.get("tunnel") or {}
    raise RuntimeError("Cloudflare did not connect in time. Studio is still running; rerun this cell to retry.")


runtime = None
host = None
process = None
key = ""
studio_url = ""
public_url = ""
HOST_LOG = Path("/kaggle/working/.strata-webui/host.log")
HOST_FILE = Path("/kaggle/working/.strata-webui/host.json")

try:
    if not RUNTIME_FILE.is_file():
        raise RuntimeError("Run 01 - Install WebUI first. %s is missing." % RUNTIME_FILE)
    runtime = json.loads(RUNTIME_FILE.read_text(encoding="utf-8"))
    for name in ("node", "webui", "dist", "source", "models", "support", "env"):
        if not runtime.get(name):
            raise RuntimeError("The runtime file is incomplete: %s is missing." % name)
    if not Path(runtime["dist"], "index.html").is_file():
        raise RuntimeError("The WebUI build is missing. Run 01 - Install WebUI again.")
    if not Path(runtime["node"]).is_file():
        raise RuntimeError("The Node executable is missing. Run 01 - Install WebUI again.")

    host = read_host_record()
    if owned_process(host):
        ui.log("HOST", "Reusing the Studio host started by this notebook (PID %s)." % host["pid"])
        studio_url = "http://127.0.0.1:%d" % int(host["studio_port"])
        key = wait_for_studio(studio_url, timeout=30)
    else:
        if host:
            ui.log("HOST", "The old host record is stale; no unrelated process was touched.")
        used = set()
        preferred = runtime.get("preferred_ports") or {}
        ports = {
            "host": pick_port(int(preferred.get("host", 8791)), used),
            "studio": pick_port(int(preferred.get("studio", 8794)), used),
            "share": pick_port(int(preferred.get("share", 8795)), used),
            "tunnel": pick_port(int(preferred.get("tunnel", 8796)), used),
        }
        environment = runtime["env"].copy()
        environment.update({
            "STRATA_PORT": str(ports["host"]),
            "STRATA_STUDIO_PORT": str(ports["studio"]),
            "STRATA_SHARE_PORT": str(ports["share"]),
            "STRATA_TUNNEL_PORT": str(ports["tunnel"]),
            "STRATA_KAGGLE_RUN_ID": uuid.uuid4().hex,
            "STRATA_EXIT_WITH_PARENT": "1",
        })
        environment.pop("STRATA_MODELS_DIR", None)
        environment.pop("STRATA_DATA_DIR", None)
        for proxy_name in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "NO_PROXY", "http_proxy", "https_proxy", "all_proxy", "no_proxy"):
            if os.environ.get(proxy_name):
                environment[proxy_name] = os.environ[proxy_name]
        studio_url = "http://127.0.0.1:%d" % ports["studio"]
        ui.set(sub="Starting the real WebUI host on loopback.", rows=[("Studio", studio_url), ("models", runtime["models"])])
        with HOST_LOG.open("ab", buffering=0) as output:
            HOST_LOG.chmod(0o600)
            process = subprocess.Popen(
                [runtime["node"], "server/index.mjs", "--studio", "--dist", runtime["dist"]],
                cwd=runtime["webui"], env=environment, stdout=output, stderr=subprocess.STDOUT,
                start_new_session=True,
            )
        host = {
            "pid": process.pid,
            "run_id": environment["STRATA_KAGGLE_RUN_ID"],
            "webui": runtime["webui"],
            "ports": ports,
            "host_port": ports["host"],
            "studio_port": ports["studio"],
            "started_at": int(time.time()),
        }
        write_host_record(host)
        key = wait_for_studio(studio_url, process=process)
        ui.log("HOST", "Studio gateway is ready on %s" % studio_url)

    if open_tunnel:
        public_url = connect_manager_tunnel(studio_url, key)
        host["public_url"] = public_url
        write_host_record(host)
    rows = [
        ("host PID", str(host["pid"])),
        ("local Studio", studio_url),
        ("models", runtime["models"]),
        ("engine + settings", runtime["support"]),
        ("host log", str(HOST_LOG)),
    ]
    ui.set(
        status="LIVE",
        sub="Studio is running. Open the private link, paste the Studio key, then use Setup and Models in the browser.",
        rows=rows,
        url=public_url,
        key=key,
        foot="This key controls the manager and changes when the host restarts. Keep it private and clear this output before saving or sharing the notebook. Inference API access is configured separately inside Studio.",
    )
    ui.log("READY", "No engine or model was downloaded by this cell.")
    if keep_alive:
        interval = max(30, int(keep_alive_interval_seconds))
        ui.set(
            sub="Studio is running. Keepalive health checks are active every %d seconds." % interval,
            foot="Interrupt this cell to stop keepalive; Studio keeps running. Run 03 - Stop to shut it down. Keep the Studio key private.",
        )
        opener = None
        try:
            while (process is None or process.poll() is None) and owned_process(host):
                time.sleep(interval)
                if process is not None and process.poll() is not None:
                    break
                if not owned_process(host):
                    break
                try:
                    if opener is None:
                        opener = login(studio_url, key)
                    state = request_json(studio_url + "/api/studio", opener=opener, timeout=15)
                    tunnel = state.get("tunnel") or {}
                    ui.log("KEEPALIVE", "Studio responded; manager tunnel: %s." % tunnel.get("status", "unknown"))
                except Exception as error:
                    opener = None
                    ui.log("KEEPALIVE", "Health check failed: %s" % clean(error))
        except KeyboardInterrupt:
            ui.set(
                status="LIVE",
                sub="Keepalive interrupted. Studio is still running.",
                foot="Run 03 - Stop when finished. Keep the Studio key private; it remains valid until the host stops.",
            )
        if (process is not None and process.poll() is not None) or not owned_process(host):
            ui.set(status="ERROR", sub="The Studio host stopped.", foot="Run 02 - Start Studio to restart it, or 03 - Stop to clear the old record.")
except KeyboardInterrupt:
    ui.set(status="STOPPED", sub="Interrupted. The host may still be running; run 03 - Stop to close it.", rows=[("host PID", str(host.get("pid")) if host else "not started")], key=key or None, url=public_url or None)
except Exception as error:
    rows = [("host PID", str(host.get("pid")) if host else "not started")]
    if studio_url:
        rows.append(("local Studio", studio_url))
    ui.set(status="ERROR", sub=clean(error) or error.__class__.__name__, rows=rows, key=key or None, url=public_url or None,
           foot="If a host PID is shown, Studio is still running locally. Fix the issue and rerun this cell, or run 03 - Stop.")


In [ ]:
# 03 - Stop
confirm_stop = True

import html
import http.cookiejar
import json
import os
import re
import signal
import textwrap
import time
import urllib.error
import urllib.request
from pathlib import Path
from IPython.display import HTML, display

RUNTIME_FILE = Path("/kaggle/working/.strata-webui/runtime.json")
HOST_FILE = Path("/kaggle/working/.strata-webui/host.json")
ANSI = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
LOGS = []


def clean(value):
    return ANSI.sub("", str(value)).replace("\r", "").rstrip()


def esc(value):
    return html.escape(str(value) if value is not None else "")


CSS = """
<style>
.st3-wrap{width:100%;box-sizing:border-box;background:#141210;color:#e8dfd0;border:1px solid #3f382c;padding:26px 30px 22px;font-family:Palatino,"Palatino Linotype",Georgia,serif;}
.st3-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;}
.st3-kicker,.st3-status{font-family:"Courier New",monospace;font-size:10px;letter-spacing:.22em;color:#b8956c;text-transform:uppercase;}
.st3-title{font-size:27px;color:#f3eadb;margin:7px 0 0;line-height:1.15;}
.st3-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.st3-sub{color:#a89880;font-size:15px;line-height:1.5;}
.st3-WORKING{color:#d2a05a;}.st3-STOPPED{color:#9fb58e;}.st3-READY{color:#9fb58e;}.st3-ERROR{color:#e18b7d;}
.st3-row{display:flex;justify-content:space-between;gap:18px;padding:5px 0;font-size:15px;line-height:1.4;}
.st3-key{color:#8a8070;min-width:150px;}.st3-value{color:#e8dfd0;text-align:right;word-break:break-word;}
.st3-console{margin-top:16px;border:1px solid #3f382c;background:#0e0c0a;}
.st3-console pre{margin:0;padding:12px 14px;font-family:"Courier New",monospace;font-size:11px;line-height:1.45;color:#bdb3a0;white-space:pre-wrap;word-break:break-word;max-height:280px;overflow:auto;}
.st3-foot{margin-top:15px;font-family:"Courier New",monospace;font-size:11px;color:#8a8070;line-height:1.5;}
</style>
"""


class Card:
    def __init__(self):
        self.status = "WORKING"
        self.sub = "Reading the Studio host record."
        self.rows = []
        self.foot = "This cell never deletes models or support files."
        self.handle = None
        self.paint()

    def paint(self):
        rows = "".join(
            '<div class="st3-row"><span class="st3-key">%s</span><span class="st3-value">%s</span></div>'
            % (esc(name), esc(value)) for name, value in self.rows
        )
        console = "".join(esc(line) + "\n" for line in LOGS[-18:])
        output = HTML(CSS + """
        <div class="st3-wrap">
          <div class="st3-top"><div class="st3-kicker">03 / Stop</div><div class="st3-status st3-%s">%s</div></div>
          <div class="st3-title">Shutdown</div><hr class="st3-rule">
          <div class="st3-sub">%s</div>
          <div style="margin-top:14px">%s</div>
          <div class="st3-console"><pre>%s</pre></div>
          <div class="st3-foot">%s</div>
        </div>""" % (esc(self.status), esc(self.status), esc(self.sub), rows, console, esc(self.foot)))
        if self.handle is None:
            self.handle = display(output, display_id="st3-stop")
        else:
            self.handle.update(output)

    def set(self, status=None, sub=None, rows=None, foot=None):
        if status is not None:
            self.status = status
        if sub is not None:
            self.sub = sub
        if rows is not None:
            self.rows = rows
        if foot is not None:
            self.foot = foot
        self.paint()

    def log(self, message):
        line = clean(message)
        if line:
            LOGS.append(textwrap.shorten("%s [STOP] %s" % (time.strftime("%H:%M:%S"), line), width=1200))
            del LOGS[:-200]
            self.paint()


ui = Card()
LOCAL_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


def request_json(url, body=None, headers=None, opener=None, timeout=20):
    request_headers = {"User-Agent": "strata-kaggle-webui/1.0"}
    data = None
    if body is not None:
        data = json.dumps(body).encode("utf-8")
        request_headers["Content-Type"] = "application/json"
    request_headers.update(headers or {})
    if request_headers.get("Origin"):
        request_headers["Origin"] = request_headers["Origin"].rstrip("/")
    request = urllib.request.Request(url, data=data, headers=request_headers, method="POST" if body is not None else "GET")
    try:
        with (opener or LOCAL_OPENER).open(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8", "replace"))
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8", "replace")
        try:
            detail = json.loads(detail).get("error", detail)
        except Exception:
            pass
        raise RuntimeError("HTTP %s: %s" % (error.code, clean(detail))) from error


def process_running(pid):
    try:
        stat = Path("/proc/%d/stat" % int(pid)).read_text(encoding="utf-8")
        return stat.rsplit(")", 1)[1].split()[0] != "Z"
    except Exception:
        return False


def owned_process(record):
    if not record or not record.get("pid") or not record.get("run_id") or not record.get("webui"):
        return False
    pid = int(record["pid"])
    if not process_running(pid):
        return False
    try:
        command = Path("/proc/%d/cmdline" % pid).read_bytes().replace(b"\0", b" ").decode("utf-8", "replace")
        environment = Path("/proc/%d/environ" % pid).read_bytes().split(b"\0")
        cwd = Path("/proc/%d/cwd" % pid).resolve()
        return (
            "server/index.mjs" in command
            and cwd == Path(record["webui"]).resolve()
            and ("STRATA_KAGGLE_RUN_ID=" + record["run_id"]).encode() in environment
        )
    except Exception:
        return False


def stop_manager_tunnel(record):
    studio_url = "http://127.0.0.1:%d" % int(record["studio_port"])
    origin = studio_url
    try:
        key = request_json(studio_url + "/studio/local-key", timeout=5)["key"]
        jar = http.cookiejar.CookieJar()
        opener = urllib.request.build_opener(urllib.request.ProxyHandler({}), urllib.request.HTTPCookieProcessor(jar))
        request_json(studio_url + "/studio/login", {"key": key}, {"Origin": origin}, opener=opener, timeout=15)
        request_json(studio_url + "/api/studio/tunnel", {"action": "stop"}, {"Origin": origin}, opener=opener, timeout=60)
        ui.log("Stopped the Studio manager tunnel through its authenticated API.")
        return True
    except Exception as error:
        ui.log("Tunnel API stop was unavailable; the host shutdown will stop it: %s" % clean(error))
        return False


def stop_owned_host(record):
    pid = int(record["pid"])
    try:
        if os.getpgid(pid) == pid:
            os.killpg(pid, signal.SIGTERM)
        else:
            os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        return True
    deadline = time.monotonic() + 20
    while time.monotonic() < deadline and process_running(pid):
        time.sleep(0.25)
    if process_running(pid):
        ui.log("The host did not exit in 20 seconds; sending SIGKILL to its process group.")
        try:
            if os.getpgid(pid) == pid:
                os.killpg(pid, signal.SIGKILL)
            else:
                os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        deadline = time.monotonic() + 5
        while time.monotonic() < deadline and process_running(pid):
            time.sleep(0.1)
    return not process_running(pid)


runtime = None
host = None
try:
    runtime = json.loads(RUNTIME_FILE.read_text(encoding="utf-8")) if RUNTIME_FILE.is_file() else {}
    host = json.loads(HOST_FILE.read_text(encoding="utf-8")) if HOST_FILE.is_file() else None
    models = runtime.get("models", "/kaggle/tmp/strata-models/Documents/Strata/Models")
    support = runtime.get("support", "/kaggle/working/.strata-webui")
    if not confirm_stop:
        ui.set(status="READY", sub="Shutdown was not confirmed. Nothing was changed.", rows=[("host", "left running")], foot="Set confirm_stop=True and run again.")
    elif not host:
        ui.set(status="STOPPED", sub="No notebook-owned Studio host is recorded.", rows=[("models", "kept"), ("support", "kept")], foot="Nothing was stopped and no files were deleted.")
    elif not owned_process(host):
        HOST_FILE.unlink(missing_ok=True)
        ui.set(status="STOPPED", sub="The recorded Studio host is already gone.", rows=[("host", "already stopped"), ("models", models), ("support", support)], foot="No unrelated process was touched. All files were kept.")
    else:
        ui.set(sub="Stopping the manager tunnel and the notebook-owned host.", rows=[("host PID", str(host["pid"])), ("models", models)])
        stop_manager_tunnel(host)
        stopped = stop_owned_host(host)
        if not stopped:
            raise RuntimeError("The Studio host process is still present after SIGKILL. Inspect the host log.")
        HOST_FILE.unlink(missing_ok=True)
        ui.log("Stopped the WebUI host, model server, and engine children.")
        ui.set(status="STOPPED", sub="Studio, its tunnel, the model server, and the engine are stopped.", rows=[
            ("host", "stopped"),
            ("manager tunnel", "stopped"),
            ("models", "kept in " + models),
            ("support", "kept in " + support),
            ("Studio key", "expired with the host process"),
        ], foot="Run 02 - Start Studio to reopen it. /kaggle/tmp is cleared when this Kaggle session ends; /kaggle/working is kept.")
except Exception as error:
    ui.set(status="ERROR", sub=clean(error) or error.__class__.__name__, rows=[("host record", str(HOST_FILE))], foot="No files were deleted. Inspect the console and the host log before trying again.")
